## 0. Import

In [1]:
import numpy as np
import pandas as pd
from PIL import Image
from math import sqrt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, Permute, Reshape, Dense,
                                     Embedding, LSTM, AdditiveAttention, Concatenate,
                                     BatchNormalization, Dropout)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

I0000 00:00:1795360295.225472   98764 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795360295.251458   98764 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795360295.807579   98764 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
IMG_SHAPE = (28, 224, 1)

PAD_TOKEN = 10
START_TOKEN = 11
END_TOKEN = 12

VOCAB_SIZE = 13  # 0-9, PAD, START, END

VERSION = "v1"  # ĐỔI TÊN VERSION Ở ĐÂY VỚI MỖI KIẾN TRÚC KHÁC NHAU

## 1. Read dataset

In [3]:
# Load dữ liệu
images = np.load("../datasets/images.npy", mmap_mode="r")
label_strings = np.load("../datasets/labels.npy")

In [4]:

# Chuyển label từ chuỗi thành danh sách các chữ số
labels = [[int(digit) for digit in label] for label in label_strings]

# Độ dài nhãn lớn nhất
max_len = max(len(label) for label in labels)

print("Number of images:", len(images))
print("Image shape:", images.shape)
print("Maximum label length:", max_len)

Number of images: 60000
Image shape: (60000, 28, 224, 1)
Maximum label length: 8


## 2. Preprocessing

In [5]:
# Padding nhãn cho Encoder:
# Mỗi nhãn cần thêm 1 vị trí dành cho END
target_len = max_len + 1

# Khởi tạo toàn bộ bằng PAD
y = np.full((len(labels), target_len), PAD_TOKEN, dtype='int32')
for i, label in enumerate(labels):
    y[i, :len(label)] = label
    y[i, len(label)] = END_TOKEN

In [6]:
# Tạo đầu vào cho Decoder:
# Dịch nhãn sang phải 1 bước và chèn START_TOKEN vào đầu
start_col = np.full((len(y), 1), START_TOKEN, dtype='int32')
dec_in = np.concatenate([start_col, y[:, :-1]], axis=1)

In [8]:
print("Maximum digit length:", max_len)
print("Target length:", target_len)

print("\nExample labels:")
for i in range(3):
    print("Original:", label_strings[i])
    print("Decoder input:", dec_in[i])
    print("Decoder target:", y[i])
    print()

Maximum digit length: 8
Target length: 9

Example labels:
Original: 6290448
Decoder input: [11  6  2  9  0  4  4  8 12]
Decoder target: [ 6  2  9  0  4  4  8 12 10]

Original: 101651
Decoder input: [11  1  0  1  6  5  1 12 10]
Decoder target: [ 1  0  1  6  5  1 12 10 10]

Original: 36731246
Decoder input: [11  3  6  7  3  1  2  4  6]
Decoder target: [ 3  6  7  3  1  2  4  6 12]



## 3. Data Split

Chia train/val/test = 70/15/15

In [9]:
idx = np.arange(len(images))
idx_train, idx_tmp = train_test_split(idx, test_size=0.30, random_state=42)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.50, random_state=42)

X_train, X_val, X_test = images[idx_train], images[idx_val], images[idx_test]
D_train, D_val, D_test = dec_in[idx_train], dec_in[idx_val], dec_in[idx_test]
y_train, y_val, y_test = y[idx_train], y[idx_val], y[idx_test]

print('Train:', X_train.shape)
print('Validation:', X_val.shape)
print('Test:', X_test.shape)

Train: (42000, 28, 224, 1)
Validation: (9000, 28, 224, 1)
Test: (9000, 28, 224, 1)


## 4. Train model

In [10]:
units = 256

# Encoder (CNN + LSTM)
img_input = Input(shape=IMG_SHAPE)          # (28, 224, 1)
x = Conv2D(32, (3, 3), padding='same', activation='relu')(img_input)    # (28, 224, 32)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)                         # (14, 112, 32)
x = Conv2D(64, (3, 3), padding='same', activation='relu')(x)            # (14, 112, 64)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)                         # (7, 56, 64)
x = Conv2D(128, (3, 3), padding='same', activation='relu')(x)           # (7, 56, 128)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)                         # (3, 28, 128)

# Chuyển feature map thành chuỗi theo chiều ngang
# feature map (H', W', C) -> chuỗi W' vector, mỗi vector mô tả một "cột" của ảnh
h, w, c = x.shape[1], x.shape[2], x.shape[3]
x = Permute((2, 1, 3))(x)
x = Reshape((w, h * c))(x)                          # (28, 3*128) = (28, 384)  

enc_out, state_h, state_c = LSTM(units, return_sequences=True, return_state=True)(x)

# Decoder (LSTM + Additive Attention)
dec_input = Input(shape=(target_len,))
emb = Embedding(VOCAB_SIZE, 64)(dec_input)
dec_out = LSTM(units, return_sequences=True)(emb, initial_state=[state_h, state_c])

context = AdditiveAttention()([dec_out, enc_out])
merged = Concatenate()([dec_out, context])
z = Dense(256, activation='relu')(merged)
outputs = Dense(VOCAB_SIZE, activation='softmax')(z)

I0000 00:00:1795360314.799040   98764 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 11753 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9


In [11]:
model = Model(inputs=[img_input, dec_input], outputs=outputs)
model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', weighted_metrics=['sparse_categorical_accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 28, 224,   │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 28, 224,   │        320 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 28, 224,   │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 14, 112,   │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 14, 112,   │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 14, 112,   │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 7, 56, 64) │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 7, 56,     │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 7, 56,     │        512 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 3, 28,     │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ permute (Permute)   │ (None, 28, 3,     │          0 │ max_pooling2d_2[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 9)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 28, 384)   │          0 │ permute[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 9, 64)     │        832 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 28, 256), │    656,384 │ reshape[0][0]     │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 9, 256)    │    328,704 │ embedding[0][0],  │
│                     │                   │            │ lstm[0][1],       │
│                     │                   │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 1,214,413 (4.63 MB)

 Trainable params: 1,213,965 (4.63 MB)

 Non-trainable params: 448 (1.75 KB)

In [12]:
# Mask những vị trí PAD
train_mask = (y_train != PAD_TOKEN).astype('float32')
val_mask = (y_val != PAD_TOKEN).astype('float32')

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

# Training
history = model.fit([X_train, D_train], y_train, 
                    sample_weight=train_mask, 
                    epochs=50, 
                    batch_size=64, 
                    verbose=1, 
                    validation_data=([X_val, D_val], y_val, val_mask), 
                    callbacks=[early_stop, reduce_lr])

Epoch 1/50


I0000 00:00:1795360317.485675   99089 cuda_dnn.cc:461] Loaded cuDNN version 92801


657/657 ━━━━━━━━━━━━━━━━━━━━ 19s 24ms/step - loss: 0.3786 - sparse_categorical_accuracy: 0.8293 - val_loss: 0.0818 - val_sparse_categorical_accuracy: 0.9700 - learning_rate: 0.0010
Epoch 2/50
657/657 ━━━━━━━━━━━━━━━━━━━━ 14s 21ms/step - loss: 0.0256 - sparse_categorical_accuracy: 0.9908 - val_loss: 0.0314 - val_sparse_categorical_accuracy: 0.9883 - learning_rate: 0.0010
Epoch 3/50
657/657 ━━━━━━━━━━━━━━━━━━━━ 15s 23ms/step - loss: 0.0135 - sparse_categorical_accuracy: 0.9948 - val_loss: 0.0214 - val_sparse_categorical_accuracy: 0.9918 - learning_rate: 0.0010
Epoch 4/50
657/657 ━━━━━━━━━━━━━━━━━━━━ 14s 22ms/step - loss: 0.0112 - sparse_categorical_accuracy: 0.9955 - val_loss: 0.0352 - val_sparse_categorical_accuracy: 0.9861 - learning_rate: 0.0010
Epoch 5/50
657/657 ━━━━━━━━━━━━━━━━━━━━ 15s 23ms/step - loss: 0.0076 - sparse_categorical_accuracy: 0.9970 - val_loss: 0.0140 - val_sparse_categorical_accuracy: 0.9948 - learning_rate: 0.0010
Epoch 6/50
461/657 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/ste

KeyboardInterrupt: 

## 5. Prediction

**THEO FILE MẪU NÀY LÀ ĐANG PREDICT THEO KIỂU AUTOREGRESSIVE, NẾU PREDICT TOÀN BỘ MỘT LẦN NHƯ CỦA HUY THÌ SẼ KHÁC, CÓ THỂ SỬA TÙY Ý**

In [ ]:
# khi dự đoán không có nhãn thật nên phải sinh từng chữ số một,
# chữ số vừa sinh ra được đưa vào làm input cho bước kế tiếp

# Khởi tạo Decoder input với toàn bộ là PAD_TOKEN, riêng vị trí đầu tiên là START_TOKEN
decoder_input = np.full((len(X_test), target_len), PAD_TOKEN, dtype='int32')
decoder_input[:, 0] = START_TOKEN

# Mảng lưu kết quả dự đoán và trạng thái hoàn thành của từng câu
y_pred = np.full((len(X_test), target_len), PAD_TOKEN, dtype='int32')
is_finished = np.zeros(len(X_test), dtype=bool)

# Dự đoán từng bước một (Greedy Decoding)
for t in range(target_len):
    probs = model.predict([X_test, decoder_input], batch_size=256, verbose=0)
    next_token = np.argmax(probs[:, t, :], axis=-1)

    # Nếu câu đã kết thúc ở các bước trước thì chỉ điền PAD_TOKEN
    next_token[is_finished] = PAD_TOKEN
    y_pred[:, t] = next_token

    # Cập nhật trạng thái cho những câu vừa sinh ra END_TOKEN
    is_finished = is_finished | (next_token == END_TOKEN)
    
    # Đưa token vừa dự đoán vào làm input cho bước tiếp theo (nếu câu chưa kết thúc)
    if t + 1 < target_len:
        not_finished = (is_finished == False)
        decoder_input[not_finished, t + 1] = next_token[not_finished]

## 6. Evaluation

In [ ]:
# Chuyển chuỗi token thành chuỗi chữ số
def decode_sequence(seq):
    digits = []
    for token in seq:
        if token >= 10:
            break
        digits.append(str(token))
    return ''.join(digits)


# Chuỗi thực tế
true_sequences = [''.join(map(str, labels[i])) for i in idx_test]

# Chuỗi dự đoán
pred_sequences = [decode_sequence(seq) for seq in y_pred]

In [ ]:
# SEQUENCE ACCURACY
correct = np.array([true == pred for true, pred in zip(true_sequences, pred_sequences)])
seq_acc = np.mean(correct)
print(f"Sequence Accuracy: {seq_acc:.4f}")

Sequence Accuracy: 0.9899


In [ ]:
# NUMERIC METRICS
# Chỉ tính trên các mẫu mà cả actual và prediction đều là chuỗi hợp lệ
valid = np.array([
    true != '' and pred != ''
    for true, pred in zip(true_sequences, pred_sequences)
])

true_num = np.array([int(true) for true in np.array(true_sequences)[valid]], dtype=np.float64)
pred_num = np.array([int(pred) for pred in np.array(pred_sequences)[valid]], dtype=np.float64)

mse = mean_squared_error(true_num, pred_num)
mae = mean_absolute_error(true_num, pred_num)
r2 = r2_score(true_num, pred_num)
# NSE
nse_denominator = np.sum((true_num - np.mean(true_num)) ** 2)
nse = 1 - np.sum((true_num - pred_num) ** 2) / nse_denominator if nse_denominator != 0 else np.nan

print(f"MAE:  {mae:.4f}")
print(f"RMSE: {sqrt(mse):.4f}")
print(f"R2:   {r2:.4f}")
print(f"NSE:  {nse:.4f}")


MAE:  7490.5826
RMSE: 442832.3042
R2:   0.9995
NSE:  0.9995


- bước preprocess ảnh nên cho vào một file scripts.py để đỡ phải chạy lại, lưu vào folder datasets/ luôn
- thống nhất toàn bộ flow code từ đầu đến cuối, chỉ khác cấu hình và kiến trúc model
- viết lại Base_Model
- model thì lưu vào models/
- mỗi notebook ở cuối file có đoạn code riêng để lưu kết quả từng model thử nghiệm ra csv vào folder output/

### Save Result

In [ ]:
output_path = "../output/<Name>Model/"           # ĐỔI TÊN Ở ĐÂY
csv_path = output_path + "<name>_model_results.csv"         # ĐỔI TÊN Ở ĐÂY
model_path = output_path + f"<name>_model_{VERSION}.keras"      # ĐỔI TÊN Ở ĐÂY

model.save(model_path)      # ĐỔI TÊN Ở ĐÂY

# Collect evaluation results
results = {
    "Model": f"<Name> Model {VERSION}",             # ĐỔI TÊN Ở ĐÂY
    "Version": VERSION,
    "Sequence_Accuracy": seq_acc,
    "MAE": mae,
    "RMSE": sqrt(mse),
    "R2": r2,
    "NSE": nse
}

results_df = pd.DataFrame([results])

if csv_path.exists():
    results_df = pd.concat([pd.read_csv(csv_path), results_df], ignore_index=True)
results_df.to_csv(csv_path, index=False)

print(f"Model saved: {model_path}")
print(f"Results saved: {csv_path}")